Cell 1 — Import libraries

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from huggingface_hub import InferenceClient

print("Libraries imported successfully!")

C:\Users\shivesh\AppData\Local\Temp\ipykernel_22912\1334016714.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\shivesh\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries imported successfully!


Cell 2 — Load your PDF

In [2]:
pdf_path = "../data/sample.pdf"

loader = PyPDFLoader(pdf_path)

documents = loader.load()

print("PDF loaded successfully!")
print("Number of pages:", len(documents))

PDF loaded successfully!
Number of pages: 4


Cell 3 — Check the PDF text

In [3]:
print(documents[0].page_content[:2000])

Assignment -1 
1. The three normative ethics traditions and their core questions are: 
           Deontology (Duty Ethics) — “What is my duty?”  
                     Consequentialism (Utilitarianism) — “What action produces the best consequences?”  
                     Virtue Ethics — “What would a good/virtuous person do? 
2. Any four features that make AI ethics distinct from “ethics as usual” are: 
Scale and impact – AI systems can aƯect millions of people quickly and across diƯerent countries.  
Autonomy and unpredictability – AI can make or inﬂuence decisions with limited direct human control, 
sometimes producing unexpected outcomes.  
Opacity (“Black Box” problem) – It can be diƯicult to understand how some AI systems arrive at their decisions.  
Responsibility and accountability – It can be unclear who is responsible when an AI system causes harm—the 
developer, company, user, or the system itself. 
3. The four layers of a sociotechnical system are: 
Technical layer — Example

Cell 4 — Split the PDF into chunks

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

Number of chunks: 13


Cell 6 — Load Hugging Face embeddings

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6368.31it/s]


Embedding model loaded successfully!


Cell 7 — Create Chroma vector database

In [6]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="pdf_rag"
)

print("Chroma vector database created successfully!")

Chroma vector database created successfully!


Cell 8 — Create the retriever

In [7]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

print("Retriever created successfully!")

Retriever created successfully!


Cell 11 — Hugging Face login

In [8]:
from huggingface_hub import login

login()

Cell 12 — Create the Hugging Face client

In [9]:
from huggingface_hub import InferenceClient

client = InferenceClient()

print("Hugging Face client created successfully!")

Hugging Face client created successfully!


Cell 14 — Create the RAG function

In [10]:
def ask_pdf(question):
    # 1. Retrieve relevant chunks
    results = retriever.invoke(question)

    # 2. Combine retrieved chunks
    context = "\n\n".join(
        doc.page_content for doc in results
    )

    # 3. Ask the LLM using only the retrieved context
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "system",
                "content": """You are a helpful PDF question-answering assistant.

Use ONLY the provided PDF context to answer the question.

If the answer cannot be found in the context, say:
"I could not find the answer in the PDF."

Do not invent information.

Give a clear and concise answer."""
            },
            {
                "role": "user",
                "content": f"""
PDF CONTEXT:

{context}

QUESTION:

{question}

ANSWER:
"""
            }
        ],
    )

    answer = response.choices[0].message.content

    # 4. Collect source page numbers
    sources = []

    for doc in results:
        page = doc.metadata.get("page")

        if page is not None:
            sources.append(page + 1)

    # Remove duplicates and sort
    sources = sorted(set(sources))

    return answer, sources

Cell 20  Interactive chatbot   

In [11]:
while True:
    question = input("\nYou: ")

    if question.lower().strip() in ["exit", "quit"]:
        print("Chatbot: Goodbye!")
        break

    answer, sources = ask_pdf(question)

    print("\nChatbot:")
    print(answer)

    print("\nSource pages:")
    print(sources)


Chatbot:
The three ethical approaches mentioned are:

1. **Deontology (Duty Ethics)**  
2. **Consequentialism (Utilitarianism)**  
3. **Virtue Ethics**

Source pages:
[1, 2, 4]

Chatbot:
Biased AI systems can cause harm by turning existing prejudices in data or processes into automated, large‑scale decisions that discriminate against particular groups.  

* **Discriminatory outcomes** – A biased hiring model, for example, may systematically reject applicants from a protected demographic, producing unfair treatment of individuals and groups.  
* **Scale and automation** – Because the AI applies the same biased logic to thousands or millions of cases without human review, the harm is amplified and can affect whole populations (group/societal‑level harm).  
* **Systemic reinforcement** – Widespread use of such biased systems can entrench and spread existing social inequalities, leading to broader structural (systemic) harm.  
* **Diffuse responsibility** – The “problem of many hands” mea